Text summarizer with spacy - yiscah mark week 6 

In [4]:
# Install and load spaCy
!pip install spacy==3.7.2
!python -m spacy download en_core_web_sm

import spacy
from string import punctuation
from spacy.lang.en.stop_words import STOP_WORDS
from heapq import nlargest

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 68.2 MB/s  0:00:00ta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


i made a corpus to make a summary out of. i used my orientation speech to mothers as in the beggining of this year, for math class. i wanted to use a corpus that means something to me so i can enjoy the results. also i see that there are alot of spelling mistakes in it so i am curious if the program will do a good job.

In [14]:
corpus = """
HI, Good evening, thank you for coming out here tonight, it’s so nice to meet all of you.
My name is Miss green, and I teach your daughters math. They are an adorable and very lively group of girls.
My goals for this class aside for the curriculum, is to have a classroom where the girls are happy and feel successful in what they are doing.
As far as the curriculum, this year we changed over to a new curriculum. We are using into math, for those familiar with go math its similar but focuses even more on word problems.  We will be learning in the book as well as with that separate booklets and handouts. We plan to iy’h cover many topics including
Linear equatons, systems of equations, basics of geometry and statistics….
Math is a subject learned through practice, which means that there is a lot of active classwork, and I try to make the lessons as exciting and interactive as possible.
The nature of math means that there are tests pretty often as well as nightly homework, they should be able to do it mostly on their own. If they are having a hard time it is important that she catches up asap because the lessons are cumulative. Encourage her to ask me for help…I go around during quiet work to answer questions..
If you have any questions comments or concerns please feel free to call me at 914-705-9925
Looking forward to a great year.

"""

we are running the corpus throuhg the spacy pipeline to give it the structure it needs to be able to be summarized.

In [6]:
nlp = spacy.load("en_core_web_sm")
doc = nlp(corpus)


This cell calculates the frequency of specific words in the model and the weight of how important theu are. stopwords are first removed becuase they are of little or no significance. this is a step to finding the main and imoortant words in the corpus.

In [7]:
word_freq = {}
for token in doc:
    if token.text.lower() not in STOP_WORDS and token.text not in punctuation:
        word = token.text.lower()
        word_freq[word] = word_freq.get(word, 0) + 1

max_freq = max(word_freq.values())
for word in word_freq:
    word_freq[word] /= max_freq

now the frequencies are summed up per sentence to see which sentences are the most central and important. that represent the main topics of the corpus.

In [15]:
sentence_scores = {}
for sent in doc.sents:
    for token in sent:
        if token.text.lower() in word_freq:
            sentence_scores[sent] = sentence_scores.get(sent, 0) + word_freq[token.text.lower()]


we choose the amount of sentences for the summary. in this case we used two. then we choose the two sentences with the highest sentence score for the summary. last we join them to form a complete summary that should encapsulate the main ideas of the corpus.

In [16]:
summary_length = 2
summary_sentences = nlargest(summary_length, sentence_scores, key=sentence_scores.get)
summary = " ".join([sent.text for sent in summary_sentences])


for more detail we also extract the key phrases in the corpus as another form of summary so we could skim the main ideas of the corpus without reading through it entirely.

In [10]:
key_phrases = set()
for chunk in doc.noun_chunks:
    phrase = chunk.text.strip().lower()
    if phrase not in STOP_WORDS and len(phrase) > 2:
        key_phrases.add(phrase)


this cell highlights named entities such as names of people and dates. this information is often very central in a corpus.

In [11]:
entities = [(ent.text, ent.label_) for ent in doc.ents]

in the cell below we get all the topic words, nouns and pronouns. this is similar to the key phrases extraction. this list of words also gives a clue to the main ideas in the corpus.

In [17]:
topic_words = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]]

in this cell we print the results of all the summaries.

In [19]:
print("=== SUMMARY ===")
print(summary)
print("\n=== KEY PHRASES ===")
print(key_phrases)
print("\n=== NAMED ENTITIES ===")
print(entities)
print("\n=== TOPIC WORDS ===")
print(topic_words)

=== SUMMARY ===
We plan to iy’h cover many topics including
Linear equatons, systems of equations, basics of geometry and statistics….
 Math is a subject learned through practice, which means that there is a lot of active classwork, and I try to make the lessons as exciting and interactive as possible.


=== KEY PHRASES ===
{'quiet work', 'a lot', 'the book', 'nightly homework', 'equations', 'questions', 'statistics', 'geometry', 'a subject', 'asap', 'active classwork', 'a new curriculum', 'girls', 'systems', 'any questions comments', 'your daughters', 'a classroom', 'many topics', 'concerns', 'my name', 'linear equatons', 'my goals', 'tests', 'the lessons', 'math', 'the nature', 'miss green', 'word problems', 'this class', 'the girls', 'a hard time', 'help', 'handouts', 'an adorable and very lively group', 'basics', 'practice', 'that separate booklets', 'a great year', 'the curriculum'}

=== NAMED ENTITIES ===
[('evening', 'TIME'), ('tonight', 'TIME'), ('this year', 'DATE'), ('Linear'

i am happy that i used my own corpus. the results are astoundingly accurate. it is so cool to see how the program was able to pull out the most important sentences as a summary. it isn't perfect, though, one small mistake i caught is that it assumed math to be a person. 
i find it so interesting the way a summary of words uses logic and coding to figure out mathematically what is most important in the corpus.